In [1]:
import requests
import pandas as pd
import uuid

# Basis configuratie gebaseerd op de technische documentatie
BASE_URL = "https://api.ah.nl"
STORE_ID = "1558"  # Dit is het unieke ID voor AH Woenselse Markt Eindhoven

# Verplichte headers om de AH app na te bootsen
headers = {
    "User-Agent": "Appie/9.28 (iPhone17,3; iPhone; CPU OS 26_1 like Mac OS X)",
    "x-application": "AHWEBSHOP",
    "x-clientname": "appie-ios",
    "x-": "9.28",
    "x-fraud-detection-installation-id": str(uuid.uuid4()), # Een unieke ID per sessie
    "Content-Type": "application/json",
    "Accept": "application/json"
}


In [2]:
def get_anonymous_token():
    auth_url = f"{BASE_URL}/mobile-auth/v1/auth/token/anonymous"
    payload = {"clientId": "appie-ios"}
    
    response = requests.post(auth_url, json=payload, headers=headers)
    response.raise_for_status() # Geeft een foutmelding als het misgaat
    
    token_data = response.json()
    return token_data['access_token']

# Activeer de sleutel voor alle volgende verzoeken
access_token = get_anonymous_token()
headers["Authorization"] = f"Bearer {access_token}"
print("Handshake succesvol: Token opgehaald.")

Handshake succesvol: Token opgehaald.


In [7]:
bargain_query = """
query GetBargains($storeId: String!) {
  bargainItems(storeId: $storeId) {
    categoryTitle  # <--- Deze voegt de categorie (zoals Vlees) toe
    product {
      title
      brand
      salesUnitSize
    }
    bargainPrice {
      priceWas
      priceNow
    }
    markdown {
      markdownPercentage
      markdownExpirationDate
    }
    stock
  }
}
"""

def fetch_laatste_kans(store_id):
    url = f"{BASE_URL}/graphql"
    
    graphql_headers = headers.copy()
    graphql_headers.update({
        "x-apollo-operation-name": "GetBargains",
        "x-apollo-operation-type": "query",
        "apollographql-client-name": "nl.ah.Appie-apollo-ios",
        "apollographql-client-version": "9.28-260102201630"
    })
    
    payload = {
        'query': bargain_query, 
        'variables': {'storeId': store_id},
        'operationName': 'GetBargains'
    }
    
    response = requests.post(url, json=payload, headers=graphql_headers)
    data = response.json()
    
    if 'errors' in data:
        print(" GraphQL Foutmelding gevonden:")
        for error in data['errors']:
            print(f" - {error.get('message')}")
        return None
        
    return data.get('data', {}).get('bargainItems')

# Haal de ruwe data opnieuw op
raw_items = fetch_laatste_kans(STORE_ID)

if raw_items:
    print(f"Succes! {len(raw_items)} producten gevonden op de Woenselse Markt.")
else:
    print("Geen data ontvangen. Controleer de output hierboven.")

Succes! 173 producten gevonden op de Woenselse Markt.


In [9]:
# Gebruik json_normalize om geneste velden (zoals product.title) plat te slaan
df_koopjes = pd.json_normalize(raw_items)

# Optioneel: Kolomnamen opschonen voor gemak
df_koopjes.columns = [c.replace('product.', '').replace('bargainPrice.', '').replace('markdown.', '') for c in df_koopjes.columns]

# Sorteer op de hoogste korting
# df = df.sort_values(by='markdownPercentage', ascending=False)

# Toon de live status
display(df_koopjes.head(10))

,categoryTitle,stock,priceWas,priceNow,markdownPercentage,markdownExpirationDate,title,brand,salesUnitSize
0,"Groente, aardappelen",2,1.99,1.19,40,2026-01-26,Bieze Rauwkost komkommer,Bieze,250 g
1,"Groente, aardappelen",5,1.39,1.04,25,2026-01-26,AH Rucola slamelange,AH,75 g
2,"Groente, aardappelen",5,1.69,1.27,25,2026-01-26,AH Aardappelen voor de stamppot,AH,1 kg
3,"Groente, aardappelen",4,1.69,1.27,25,2026-01-26,AH Hutspot,AH,500 g
4,"Groente, aardappelen",4,2.99,2.24,25,2026-01-26,AH Roerbakgroente Italiaans fijngesneden,AH,400 g
5,"Groente, aardappelen",3,1.99,1.49,25,2026-01-26,AH Taugé grootverpakking,AH,250 g
6,"Groente, aardappelen",3,2.39,1.79,25,2026-01-26,AH Biologisch Vastkokende aardappelen,AH Biologisch,1 kg
7,"Groente, aardappelen",2,3.99,1.79,25,2026-01-26,AH Stoomgroente bloemkool marscarponesaus,AH,450 g
8,"Groente, aardappelen",2,1.59,1.19,25,2026-01-26,AH Eikenblad slamelange,AH,100 g
9,"Groente, aardappelen",2,4.49,3.37,25,2026-01-26,AH Pompoensoep verspakket,AH,6 pers | 35 min


In [60]:


def get_df_bonus():
    """Haalt landelijke bonus op en geeft een schoon pandas DataFrame terug"""
    token = get_ah_token()
    auth_headers = {**HEADERS, "Authorization": f"Bearer {token}"}
    
    # GraphQL query gebaseerd op het schema: bonusPromotions -> products -> priceV2
    query = """
    query GetNationalBonus {
      bonusPromotions {
        title
        products {
          title
          brand
          priceV2 {
            now { amount }
            was { amount }
            discount { description }
          }
        }
      }
    }
    """
    
    response = requests.post(f"{BASE_URL}/graphql", json={"query": query}, headers=auth_headers)
    response.raise_for_status()
    raw_data = response.json().get('data', {}).get('bonusPromotions', [])
    
    if not raw_data:
        return pd.DataFrame()

    # Transformatie: we 'flatten' de producten uit de promotiegroepen
    df = pd.json_normalize(
        raw_data, 
        record_path=['products'], 
        meta=['title'],
        record_prefix='product_',
        meta_prefix='promo_'
    )
    
    # Selectie en hernoemen van kolommen voor een schoon resultaat
    mapping = {
        'product_title': 'Product',
        'product_brand': 'Merk',
        'product_priceV2.now.amount': 'Prijs_Nu',
        'product_priceV2.was.amount': 'Prijs_Was',
        'product_priceV2.discount.description': 'Bonus_Tekst',
        'promo_title': 'Categorie'
    }
    
    # Alleen de kolommen die we willen hebben
    df_bonus = df.rename(columns=mapping)[list(mapping.values())]
    
    # Kleine extra opschoning: bereken het voordeel in euro's
    df_bonus['Korting_Euro'] = (df_bonus['Prijs_Was'] - df_bonus['Prijs_Nu']).round(2)
    
    return df_bonus

# --- UITVOERING ---
df_bonus = get_df_bonus()
# Check het resultaat
if not df_bonus.empty:
    print(f"✅ Gelukt! {len(df_bonus)} bonusitems geladen.")
    # Sorteer op de hoogste korting in euro's
    df_bonus = df_bonus.sort_values(by='Korting_Euro', ascending=False)
    print(df_bonus.head(10))
else:
    print("❌ Geen bonusdata gevonden.")

✅ Gelukt! 3514 bonusitems geladen.
                          Product   Merk  Prijs_Nu  Prijs_Was  Bonus_Tekst  \
3026   Nomad Fleece heren maat XL  Nomad     19.99      39.98  50% korting   
3027    Nomad Fleece heren maat L  Nomad     19.99      39.98  50% korting   
3029    Nomad Fleece dames maat S  Nomad     19.99      39.98  50% korting   
3030    Nomad Fleece dames maat M  Nomad     19.99      39.98  50% korting   
3028   Nomad Fleece dames maat XL  Nomad     19.99      39.98  50% korting   
3031         Nomad Fleece dames L  Nomad     19.99      39.98  50% korting   
3025    Nomad Fleece heren maat M  Nomad     19.99      39.98  50% korting   
3024  Nomad Fleece heren maat XXL  Nomad     19.99      39.98  50% korting   
3003    Nomad Ski handschoen L/XL  Nomad     18.99      37.98  50% korting   
3002     Nomad Ski handschoen S/M  Nomad     18.99      37.98  50% korting   

                Categorie  Korting_Euro  
3026  Nomad Thermokleding         19.99  
3027  Nomad Thermokled

In [63]:
import pandas as pd
import re

def parse_ah_bonus(df):
    def calculate_deal(row):
        text = str(row['Bonus_Tekst']).lower()
        now = row['Prijs_Nu']
        was = row['Prijs_Was']
        
        # 1. Percentage (bijv. 50% korting)
        pct_match = re.search(r'(\d+)%\s*(?:korting|op=op)', text)
        if pct_match:
            pct = float(pct_match.group(1)) / 100
            # Als was-prijs ontbreekt, reken hem terug
            inferred_was = was if not pd.isna(was) else (now / (1 - pct))
            return 'PERCENTAGE', pct * 100, inferred_was

        # 2. X + Y gratis (bijv. 1 + 1 gratis, 2 + 1 gratis)
        plus_match = re.search(r'(\d+)\s*\+\s*(\d+)\s*gratis', text)
        if plus_match:
            x, y = float(plus_match.group(1)), float(plus_match.group(2))
            pct = y / (x + y)
            return 'X+Y GRATIS', pct * 100, (now * (x + y) / x)

        # 3. 2e gratis of 2e halve prijs
        if '2e gratis' in text:
            return '1+1 GRATIS', 50.0, (now * 2)
        if '2e halve prijs' in text:
            return '25% KORTING', 25.0, (now * 2 / 1.5)

        # 4. Bundel deals (bijv. 2 voor 5.00)
        voor_match = re.search(r'(\d+)\s*voor\s*(\d+\.?\d*)', text)
        if voor_match:
            count, total_price = float(voor_match.group(1)), float(voor_match.group(2))
            if not pd.isna(was):
                pct = (1 - (total_price / (was * count))) * 100
                return 'BUNDEL DEAL', pct, was
            return 'BUNDEL DEAL', 0.0, now

        return 'OVERIG', 0.0, was

    # Pas de logica toe
    results = df.apply(calculate_deal, axis=1, result_type='expand')
    df['Discount_Type'] = results[0]
    df['Korting_Pct'] = results[1]
    df['Prijs_Was'] = df['Prijs_Was'].fillna(results[2])
    
    # Bereken de uiteindelijke korting in Euro's
    df['Korting_Euro'] = (df['Prijs_Was'] - df['Prijs_Nu']).round(2)
    
    return df

# Run de sniper
df_bonus = parse_ah_bonus(df_bonus)

In [71]:
print(df_bonus.columns)
print(df_koopjes.columns)

Index(['Product', 'Merk', 'Prijs_Nu', 'Prijs_Was', 'Bonus_Tekst', 'Categorie',
       'Korting_Euro', 'Discount_Type', 'Korting_Pct'],
      dtype='object')
Index(['categoryTitle', 'stock', 'priceWas', 'priceNow', 'markdownPercentage',
       'markdownExpirationDate', 'title', 'brand', 'salesUnitSize'],
      dtype='object')


In [75]:
# 1. Normaliseer de titels zoals we al deden
df_bonus['Product_clean'] = df_bonus['Product'].str.lower().str.strip()
df_koopjes['title_clean'] = df_koopjes['title'].str.lower().str.strip()

# 2. De Merge
df_double_deals = pd.merge(
    df_bonus, 
    df_koopjes, 
    left_on='Product_clean', 
    right_on='title_clean', 
    how='inner'
)

# 3. FIX: Zet prijzen om naar getallen (errors='coerce' maakt van tekst NaN)
df_double_deals['priceNow'] = pd.to_numeric(df_double_deals['priceNow'], errors='coerce')
df_double_deals['Prijs_Was'] = pd.to_numeric(df_double_deals['Prijs_Was'], errors='coerce')

# 4. Bereken nu de korting (met beveiliging tegen delen door nul)
df_double_deals['Totale_Korting_Percentage'] = (1 - (df_double_deals['priceNow'] / df_double_deals['Prijs_Was'])) * 100

# 5. Filter en toon resultaat
sniper_resultaat = df_double_deals[[
    'Product', 
    'Bonus_Tekst', 
    'markdownPercentage', 
    'Prijs_Was', 
    'priceNow', 
    'Totale_Korting_Percentage'
]].dropna(subset=['Totale_Korting_Percentage']) # Haal ongeldige berekeningen eruit


df_double_deals['Totale_Korting_Percentage'] = df_double_deals['Totale_Korting_Percentage'].round(2)
display(sniper_resultaat.sort_values(by='Totale_Korting_Percentage', ascending=False))

,Product,Bonus_Tekst,markdownPercentage,Prijs_Was,priceNow,Totale_Korting_Percentage
16,Melkunie Protein bosbes kwark,2e gratis,25,3.18,1.19,62.578616
15,AH Scharrel kipburger 2 stuks,2e gratis,25,7.18,2.69,62.534819
17,Campina Zacht & luchtig vanille smaak,2e gratis,25,4.98,1.87,62.449799
0,Johma Oudekaas-pesto salade,30% korting,25,3.69,1.94,47.425474
3,Friesche Vlag Barista haver,NaN,40,2.69,1.61,40.148699
1,AH Scharrel kipfilet blokjes,1 euro korting,25,5.49,3.37,38.615665
4,Optimel Drinkyoghurt mango-passievrucht,NaN,25,1.59,1.19,25.157233
10,AH Luxe wok Japans,2 voor 3.99,25,3.59,2.69,25.069638
2,L'Atelier Melkchocoladereep hazelnoot & rozijn,NaN,25,4.39,3.29,25.056948
13,AH Verse maaltijd roti kip,2 VOOR 12.00,25,6.99,5.24,25.035765


In [99]:
import os
from dotenv import load_dotenv
import google.generativeai as genai

# 1. Laad de variabelen uit je .env bestand
load_dotenv()

# 2. Haal de specifieke key op
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

print(GEMINI_API_KEY)
# 3. Controleer of de key aanwezig is
if not GEMINI_API_KEY:
    raise ValueError("Gemini API Key is niet gevonden in het .env bestand. Controleer of de naam in .env exact 'GEMINI_API_KEY' is.")
else:
    # GLOBAAL CONFIGUREREN 
    genai.configure(api_key=GEMINI_API_KEY)
    print("Gemini API Key is geladen uit .env en geconfigureerd.")

AIzaSyB0pRQymyAL5aHRTbN2p2Df34LKrnD6ox
Gemini API Key is geladen uit .env en geconfigureerd.


In [100]:

model = genai.GenerativeModel("gemma-3-27b-it")



In [101]:
# Simuleer een lijst met sniper-toppers (Bargains + Bonus)
test_deals = """
| Product | Korting | Deal Type |
| :--- | :--- | :--- |
| AH Half om half gehakt | 70% | Sticker |
| Biodermal Dagcrème | 50% | 1+1 Gratis |
| Appelsientje Jus d'Orange | 40% | 3+2 Gratis |
| Nomad Fleece Vest | 50% | Percentage |
"""

test_prompt = f"""
Je bent de 'AH Sniper Chef'. Jouw missie: verzin een creatief recept met de onderstaande koopjes.
Regels:
1. Gebruik minimaal 2 items uit de lijst.
2. Vul aan met basisvoorraad (olie, kruiden, rijst, etc.).
3. Wees eerlijk: als de ingrediënten niet bij elkaar passen (zoals fleecevesten en gehakt), negeer dan het non-food item.
4. Geef het recept een stoere naam.

De Koopjes:
{test_deals}
"""

# Test de aanroep
response = model.generate_content(test_prompt)
print(response.text)

InvalidArgument: 400 API key not valid. Please pass a valid API key. [reason: "API_KEY_INVALID"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "generativelanguage.googleapis.com"
}
, locale: "en-US"
message: "API key not valid. Please pass a valid API key."
]